# Purpose
This notebook conducts statistical analysis and generates processed data underlying figures that demonstrate model behavior.

In [ ]:
import pickle
import numpy as np
from scipy import stats
import pandas as pd
import sys
sys.path.append("..")
from scripts import stats_analysis, indices, directed_modularity

In [ ]:
cond_ls = ['A', 'C'] # Weight adjustment condition
eta = 10.0 # learning rate
K = 30 # number of instances

r2_thresh = 0.85 # threshold of r2 for the power-law fit
num_bins = 20 # number of bins on the logscale for the histogram of the upper tail of the weight distribution

Nsim = 100 # number of null networks for connectivity metrics

# Control condition 1: adaptive weight adjustment with fixed connectivity

In [ ]:
# Import resulted networks at the end of simulations
tau_ls = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
A_wAdjAlone = {}
for cond in cond_ls:
    A_wAdjAlone[cond] = {}
    for tau in tau_ls:
        A_wAdjAlone[cond][tau] = {}
        for k in range(K):
            f = open('./Output/wAdjust_alone/'+cond+'_eta_'+str(eta)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
            A_wAdjAlone[cond][tau][k], _ = pickle.load(f)
            f.close()

In [ ]:
# Compare candidate distributions for the best fit to the simulated weights
KS_stats = {}; wilcoxon_p = {}
for cond in cond_ls:
    KS_stats[cond] = {}; wilcoxon_p[cond] = {}
    for tau in tau_ls:
        KS_stats[cond][tau] = np.zeros((K, 7))
        wilcoxon_p[cond][tau] = {}
        for k in range(K):
            # KS statistics of candidate distributions
            KS_stats[cond][tau][k, :] = stats_analysis.ks_stats(A_wAdjAlone[cond][tau][k])

        # find the distribution whose KS statistics significantly smaller than others
        wilcoxon_p[cond][tau] = stats_analysis.best_ks(KS_stats[cond][tau])

f = open('./Output/wAdjust_alone/KS_stats.pckl', 'wb')
pickle.dump([KS_stats, wilcoxon_p], f)
f.close()

In [ ]:
tau = 0.05
wb_params = {}
for s, cond in enumerate(cond_ls):
    wb_params[cond] = {}
    c = np.zeros(K); scale = np.zeros(K)
    for k in range(K):
        weights = A_wAdjAlone[cond][tau][k][A_wAdjAlone[cond][tau][k]>0]
        c[k], loc, scale[k] = stats.weibull_min.fit(weights, floc = 0)
    wb_params[cond]['loc'] = np.mean(c)
    wb_params[cond]['scale'] = np.std(c)

f = open('./Output/wAdjust_alone/weibull_fit_tau_'+str(tau)+'.pckl', 'wb')
pickle.dump(wb_params, f)
f.close()

In [ ]:
# Compute the decay rate of the right tail of the weight distribution
res_PL = {}; PL_expon = {}
for cond in cond_ls:
    res_PL[cond] = {}; PL_expon[cond] = {}

    for tau in tau_ls:
        res_PL_temp = {}
        PL_expon_temp = []
        for k in range(K):
            # fit the right tail of the weight distribution with a linear function
            res_PL_temp[k] = stats_analysis.PL_MLE(A_wAdjAlone[cond][tau][k], num_bins)

            # keep the decay rate if r2 of the linear fit is above the threshold
            if res_PL_temp[k][2] > r2_thresh:
                PL_expon_temp.append(-res_PL_temp[k][1])

        res_PL[cond][tau] = pd.DataFrame.from_dict(res_PL_temp, orient='index', columns = ["intercept", "slope", "r2"])

        # keep the decay rates if more than half of the instances have r2 above the threshold
        if np.mean(res_PL[cond][tau]['r2'] > r2_thresh) > 0.5:
            PL_expon[cond][tau] = PL_expon_temp

f = open('./Output/wAdjust_alone/PL_fit.pckl', 'wb')
pickle.dump([res_PL, PL_expon], f)
f.close()

In [ ]:
# Compute normalized connectivity metrics
cc_null_norm = {}; ae_null_norm = {}; sw_null_norm = {}; mod_null_norm = {}
null_A = {}

for cond in cond_ls:
    cc_null_norm[cond] = {}
    ae_null_norm[cond] = {}
    sw_null_norm[cond] = {}
    mod_null_norm[cond] = {}
    null_A[cond] = {}

    for tau in tau_ls:
        cc_null_norm[cond][tau] = np.zeros(K)
        ae_null_norm[cond][tau] = np.zeros(K)
        mod_null_norm[cond][tau] = np.zeros(K)
        null_A[cond][tau] = {}
        for k in range(K):
            A = A_wAdjAlone[cond][tau][k].copy()

            # raw values of connectivity metrics
            clust_coef = indices.cluster_coefficient(A)
            ave_eff = indices.average_efficiency(A)
            mod, c = directed_modularity.getModularityIndex(A)

            # raw values of connectivity metrics of the null networks
            null_A[cond][tau][k] = {}
            null_cc_temp = np.zeros(Nsim)
            null_ave_eff_temp = np.zeros(Nsim)
            null_mod_temp = np.zeros(Nsim)
            for i in range(Nsim):
                # Generate null networks for connectivity metrics by permuting weights
                null_A[cond][tau][k][i] = indices.randomizeWeightsKeepTopology(A)
                null_cc_temp[i] = indices.cluster_coefficient(null_A[cond][tau][k][i])
                null_ave_eff_temp[i] = indices.average_efficiency(null_A[cond][tau][k][i])
                null_mod_temp[i], c = directed_modularity.getModularityIndex(null_A[cond][tau][k][i])

            # normalized connectivity metrics
            cc_null_norm[cond][tau][k] = clust_coef/np.mean(null_cc_temp)
            ae_null_norm[cond][tau][k] = ave_eff/np.mean(null_ave_eff_temp)
            mod_null_norm[cond][tau][k] = mod/np.mean(null_mod_temp)

        sw_null_norm[cond][tau] = cc_null_norm[cond][tau] * ae_null_norm[cond][tau]

f = open('./Output/wAdjust_alone/Weighted_Null.pckl', 'wb')
pickle.dump(null_A, f)
f.close()

f = open('./Output/wAdjust_alone/Topo_metrics.pckl', 'wb')
pickle.dump([cc_null_norm, ae_null_norm, sw_null_norm, mod_null_norm], f)
f.close()

In [ ]:
# Import dictionaries of networks sampled at different steps in simulations for stationary analysis
tau_ls = [0.05, 0.1, 0.5]
A_wAdjAlone = {}
for cond in cond_ls:
    A_wAdjAlone[cond] = {}
    for tau in tau_ls:
        A_wAdjAlone[cond][tau] = {}
        for k in range(K):
            f = open('./Output_long/wAdjust_alone/'+cond+'_eta_'+str(eta)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
            A_wAdjAlone[cond][tau][k], _ = pickle.load(f)
            f.close()

In [ ]:
# Summary statistics
std = {}; skew = {}; kurt = {}
for cond in cond_ls:
    std[cond] = {}; skew[cond] = {}; kurt[cond] = {}
    for tau in tau_ls:
        std[cond][tau] = {}
        skew[cond][tau] = {}
        kurt[cond][tau] = {}
        for i, j in enumerate(A_wAdjAlone[cond][tau][0].keys()):
            std[cond][tau][j] = np.zeros(K)
            skew[cond][tau][j] = np.zeros(K)
            kurt[cond][tau][j] = np.zeros(K)

for cond in cond_ls:
    for tau in tau_ls:
        for k in range(K):
            for i, j in enumerate(A_wAdjAlone[cond][tau][k].keys()):
                std[cond][tau][j][k], skew[cond][tau][j][k], kurt[cond][tau][j][k] = stats_analysis.stats_summaries(A_wAdjAlone[cond][tau][k][j])
                
f = open('./Output_long/wAdjust_alone/summary_stats_evol.pckl', 'wb')
pickle.dump([std, skew, kurt], f)
f.close()

In [ ]:
# Connectivity metrics
cc = {}; ae = {}; mod = {}

for cond in cond_ls:
    cc[cond] = {}
    ae[cond] = {}
    mod[cond] = {} 
    for tau in tau_ls:
        cc[cond][tau] = {}
        ae[cond][tau] = {}
        mod[cond][tau] = {}
        for i, j in enumerate(A_wAdjAlone[cond][tau][0].keys()):
            cc[cond][tau][j] = np.zeros(K)
            ae[cond][tau][j] = np.zeros(K)
            mod[cond][tau][j] = np.zeros(K)

for cond in cond_ls:
    for tau in tau_ls:
        for k in range(K):
            for i, j in enumerate(A_wAdjAlone[cond][tau][k].keys()):
                A = A_wAdjAlone[cond][tau][k][j].copy()
                cc[cond][tau][j][k] = indices.cluster_coefficient(A)
                ae[cond][tau][j][k] = indices.average_efficiency(A)
                mod[cond][tau][j][k], _ = directed_modularity.getModularityIndex(A)

f = open('./Output_long/wAdjust_alone/Topo_metrics_evol.pckl', 'wb')
pickle.dump([cc, ae, mod], f)
f.close()  

# Control condition 2: Random rewiring + adaptive weight adjustment

In [ ]:
# Import resulted networks at the end of simulations
tau_ls = [0.05, 0.1, 0.2, 0.5, 1.0]
A_randAdapt = {}
for cond in cond_ls:
    A_randAdapt[cond] = {}
    for tau in tau_ls:
        A_randAdapt[cond][tau] = {}
        for k in range(K):
            f = open('./Output/wAdjust_randRew/'+cond+'_eta_'+str(eta)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
            A_randAdapt[cond][tau][k], _ = pickle.load(f)
            f.close()

In [ ]:
# Compute the decay rate of the right tail of the weight distribution
res_PL = {}; PL_expon = {}
for cond in cond_ls:
    res_PL[cond] = {}; PL_expon[cond] = {}

    for tau in tau_ls:
        res_PL_temp = {}
        PL_expon_temp = []
        for k in range(K):
            # fit the right tail of the weight distribution with a linear function
            res_PL_temp[k] = stats_analysis.PL_MLE(A_randAdapt[cond][tau][k], num_bins)
            # keep the decay rate if r2 of the linear fit is above the threshold
            if res_PL_temp[k][2] > r2_thresh:
                PL_expon_temp.append(-res_PL_temp[k][1])

        res_PL[cond][tau] = pd.DataFrame.from_dict(res_PL_temp, orient='index', columns = ["intercept", "slope", "r2"])
        
        # keep the decay rates if more than half of the instances have r2 above the threshold
        if np.mean(res_PL[cond][tau]['r2'] > r2_thresh) > 0.5:
            PL_expon[cond][tau] = PL_expon_temp

f = open('./Output/wAdjust_randRew/PL_fit.pckl', 'wb')
pickle.dump([res_PL, PL_expon], f)
f.close()

In [ ]:
# Compute normalized connectivity metrics
cc_null_norm = {}; ae_null_norm = {}; sw_null_norm = {}; mod_null_norm = {}
null_A = {}

for cond in cond_ls:
    cc_null_norm[cond] = {}
    ae_null_norm[cond] = {}
    sw_null_norm[cond] = {}
    mod_null_norm[cond] = {}
    null_A[cond] = {}

    for tau in tau_ls:
        cc_null_norm[cond][tau] = np.zeros(K)
        ae_null_norm[cond][tau] = np.zeros(K)
        mod_null_norm[cond][tau] = np.zeros(K)
        null_A[cond][tau] = {}
        for k in range(K):
            A = A_wAdjAlone[cond][tau][k].copy()

            # raw values of connectivity metrics
            clust_coef = indices.cluster_coefficient(A)
            ave_eff = indices.average_efficiency(A)
            mod, c = directed_modularity.getModularityIndex(A)

            # raw values of connectivity metrics of the null networks
            null_A[cond][tau][k] = {}
            null_cc_temp = np.zeros(Nsim)
            null_ave_eff_temp = np.zeros(Nsim)
            null_mod_temp = np.zeros(Nsim)
            for i in range(Nsim):
                # Generate null networks for connectivity metrics by permuting weights
                null_A[cond][tau][k][i] = indices.randomizeWeightsKeepTopology(A)
                null_cc_temp[i] = indices.cluster_coefficient(null_A[cond][tau][k][i])
                null_ave_eff_temp[i] = indices.average_efficiency(null_A[cond][tau][k][i])
                null_mod_temp[i], c = directed_modularity.getModularityIndex(null_A[cond][tau][k][i])

            # normalized connectivity metrics
            cc_null_norm[cond][tau][k] = clust_coef/np.mean(null_cc_temp)
            ae_null_norm[cond][tau][k] = ave_eff/np.mean(null_ave_eff_temp)
            mod_null_norm[cond][tau][k] = mod/np.mean(null_mod_temp)

        sw_null_norm[cond][tau] = cc_null_norm[cond][tau] * ae_null_norm[cond][tau]

f = open('./Output/wAdjust_randRew/Weighted_Null.pckl', 'wb')
pickle.dump(null_A, f)
f.close()
f = open('./Output/wAdjust_randRew/Topo_metrics.pckl', 'wb')
pickle.dump([cc_null_norm, ae_null_norm, sw_null_norm, mod_null_norm], f)
f.close()

In [ ]:
# Import dictionaries of networks sampled at different steps in simulations for stationary analysis
tau_ls = [0.05, 0.1, 0.5]
A_randAdapt = {}
for cond in cond_ls:
    A_randAdapt[cond] = {}
    for tau in tau_ls:
        A_randAdapt[cond][tau] = {}
        for k in range(K):
            f = open('./Output_long/wAdjust_randRew/'+cond+'_eta_'+str(eta)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
            A_randAdapt[cond][tau][k], _ = pickle.load(f)
            f.close()

In [ ]:
# Summary statistics
std = {}; skew = {}; kurt = {}
for cond in cond_ls:
    std[cond] = {}; skew[cond] = {}; kurt[cond] = {}
    for tau in tau_ls:
        std[cond][tau] = {}
        skew[cond][tau] = {}
        kurt[cond][tau] = {}
        for i, j in enumerate(A_randAdapt[cond][tau][0].keys()):
            std[cond][tau][j] = np.zeros(K)
            skew[cond][tau][j] = np.zeros(K)
            kurt[cond][tau][j] = np.zeros(K)

for cond in cond_ls:
    for tau in tau_ls:
        for k in range(K):
            for i, j in enumerate(A_randAdapt[cond][tau][k].keys()):
                std[cond][tau][j][k], skew[cond][tau][j][k], kurt[cond][tau][j][k] = stats_analysis.stats_summaries(A_randAdapt[cond][tau][k][j])
         
f = open('./Output_long/wAdjust_randRew/summary_stats_evol.pckl', 'wb')
pickle.dump([std, skew, kurt], f)
f.close()      

In [ ]:
# Connectivity metrics
cc = {}; ae = {}; mod = {}

for cond in cond_ls:
    cc[cond] = {}
    ae[cond] = {}
    mod[cond] = {} 
    for tau in tau_ls:
        cc[cond][tau] = {}
        ae[cond][tau] = {}
        mod[cond][tau] = {}
        for i, j in enumerate(A_randAdapt[cond][tau][0].keys()):
            cc[cond][tau][j] = np.zeros(K)
            ae[cond][tau][j] = np.zeros(K)
            mod[cond][tau][j] = np.zeros(K)

for cond in cond_ls:
    for tau in tau_ls:
        for k in range(K):
            for i, j in enumerate(A_randAdapt[cond][tau][k].keys()):
                A = A_randAdapt[cond][tau][k][j].copy()
                cc[cond][tau][j][k] = indices.cluster_coefficient(A)
                ae[cond][tau][j][k] = indices.average_efficiency(A)
                mod[cond][tau][j][k], _ = directed_modularity.getModularityIndex(A)

f = open('./Output_long/wAdjust_randRew/Topo_metrics_evol.pckl', 'wb')
pickle.dump([cc, ae, mod], f)
f.close()  

# Dual adaptive algorithm

## Networks generated at different $\tau_{reweight}$ and $p_{in} = 0.5$.

In [ ]:
# Import resulted networks at the end of simulations
pin = 0.5
tau_ls = [0.05, 0.1, 0.2, 0.5, 1.0]
A_matrices = {}
for cond in cond_ls:
    A_matrices[cond] = {}
    for tau in tau_ls:
        A_matrices[cond][tau] = {}
        for k in range(K):       
            f = open('./Output/dualAdapt/'+cond+'_pin_'+str(pin)+'_eta_'+str(eta)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
            A_matrices[cond][tau][k], _ = pickle.load(f)
            f.close()

In [ ]:
# Compare candidate distributions for the best fit to the simulated weights
KS_stats = {}; wilcoxon_p = {}
for cond in cond_ls:
    KS_stats[cond] = {}; wilcoxon_p[cond] = {}
    for tau in tau_ls:
        KS_stats[cond][tau] = np.zeros((K, 7))
        wilcoxon_p[cond][tau] = {}
        for k in range(K):
            # KS statistics of candidate distributions
            KS_stats[cond][tau][pin][k, :] = stats_analysis.ks_stats(A_matrices[cond][tau][pin][k])

        # find the distribution whose KS statistics significantly smaller than others
        wilcoxon_p[cond][tau] = stats_analysis.best_ks(KS_stats[cond][tau])

f = open('./Output/dualAdapt/KS_stats_pin_'+str(pin)+'.pckl', 'wb')
pickle.dump([KS_stats, wilcoxon_p], f)
f.close()

In [ ]:
# Compute the decay rate of the right tail of the weight distribution
res_PL = {}; PL_expon = {} 
for cond in cond_ls:
    res_PL[cond] = {}; PL_expon[cond] = {} 
    
    for tau in tau_ls:
        res_PL_temp = {} 
        PL_expon_temp = []
        for k in range(K):
            # fit the right tail of the weight distribution with a linear function
            res_PL_temp[k] = stats_analysis.PL_MLE(A_matrices[cond][tau][k], num_bins)
            
            # keep the decay rate if r2 of the linear fit is above the threshold
            if res_PL_temp[k][2] > r2_thresh:
                PL_expon_temp.append(-res_PL_temp[k][1])
                
        res_PL[cond][tau] = pd.DataFrame.from_dict(res_PL_temp, orient='index', columns = ["intercept", "slope", "r2"])
        
        # keep the decay rates if more than half of the instances have r2 above the threshold
        if np.mean(res_PL[cond][tau]['r2'] > r2_thresh) > 0.5:
            PL_expon[cond][tau] = PL_expon_temp

f = open('./Output/dualAdapt/PL_fit_pin_'+str(pin)+'.pckl', 'wb')
pickle.dump([res_PL, PL_expon], f)
f.close()

In [ ]:
# Import dictionaries of networks sampled at different steps in simulations for stationary analysis
tau_ls = [0.05, 0.1, 0.5]

A_dualAdapt = {}
for cond in cond_ls:
    A_dualAdapt[cond] = {}
    for tau in tau_ls:
        A_dualAdapt[cond][tau] = {}
        for k in range(K):       
            f = open('./Output_long/dualAdapt/'+cond+'_pin_'+str(pin)+'_eta_'+str(eta)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
            A_dualAdapt[cond][tau][k], _ = pickle.load(f)
            f.close()

In [ ]:
# Summary statistics
std = {}; skew = {}; kurt = {}
for cond in cond_ls:
    std[cond] = {}; skew[cond] = {}; kurt[cond] = {}
    for tau in tau_ls:
        std[cond][tau] = {}
        skew[cond][tau] = {}
        kurt[cond][tau] = {}
        for i, j in enumerate(A_dualAdapt[cond][tau][0].keys()):
            std[cond][tau][j] = np.zeros(K)
            skew[cond][tau][j] = np.zeros(K)
            kurt[cond][tau][j] = np.zeros(K)
            
for cond in cond_ls:
    for tau in tau_ls:
        for k in range(K):
            for i, j in enumerate(A_dualAdapt[cond][tau][k].keys()):
                std[cond][tau][j][k], skew[cond][tau][j][k], kurt[cond][tau][j][k] = stats_analysis.stats_summaries(A_dualAdapt[cond][tau][k][j])
         
f = open('./Output_long/dualAdapt/summary_stats_evol_pin_'+str(pin)+'.pckl', 'wb')
pickle.dump([std, skew, kurt], f)
f.close()

In [ ]:
# Connectivity metrics
cc = {}; ae = {}; mod = {}

for cond in cond_ls:
    cc[cond] = {}
    ae[cond] = {}
    mod[cond] = {} 
    for tau in tau_ls:
        cc[cond][tau] = {}
        ae[cond][tau] = {}
        mod[cond][tau] = {}
        for i, j in enumerate(A_dualAdapt[cond][tau][0].keys()):
            cc[cond][tau][j] = np.zeros(K)
            ae[cond][tau][j] = np.zeros(K)
            mod[cond][tau][j] = np.zeros(K)

for cond in cond_ls:
    for tau in tau_ls:
        for k in range(K):
            for i, j in enumerate(A_dualAdapt[cond][tau][k].keys()):
                A = A_dualAdapt[cond][tau][k][j].copy()
                cc[cond][tau][j][k] = indices.cluster_coefficient(A)
                ae[cond][tau][j][k] = indices.average_efficiency(A)
                mod[cond][tau][j][k], _ = directed_modularity.getModularityIndex(A)

f = open('./Output_long/dualAdapt/Topo_metrics_evol_pin_'+str(pin)+'.pckl', 'wb')
pickle.dump([cc, ae, mod], f)
f.close()  

## Networks generated at different $p_{in}$.

In [ ]:
# Import resulted networks at the end of simulations
tau_ls = [0.05, 0.1, 0.5]
pin_ls = [0.1, 0.3, 0.5, 0.7, 0.9]
A_matrices = {}
for cond in cond_ls:
    A_matrices[cond] = {}
    for tau in tau_ls:
        A_matrices[cond][tau] = {}
        for pin in pin_ls:
            A_matrices[cond][tau][pin] = {}
            for k in range(K):       
                f = open('./Output/dualAdapt/'+cond+'_pin_'+str(pin)+'_eta_'+str(eta)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
                A_matrices[cond][tau][pin][k], _ = pickle.load(f)
                f.close()

In [ ]:
# Compare candidate distributions for the best fit to the simulated weights at tau_reweight = 0.1
tau = 0.1
for cond in cond_ls:
    KS_stats[cond] = {}; wilcoxon_p[cond] = {}
    
    for pin in pin_ls:
        KS_stats[cond][pin] = np.zeros((K, 7))
        wilcoxon_p[cond][pin] = {}
        for k in range(K):
            # KS statistics of candidate distributions
            KS_stats[cond][pin][k, :] = stats_analysis.ks_stats(A_matrices[cond][tau][pin][k])

        # find the distribution whose KS statistics significantly smaller than others
        wilcoxon_p[cond][pin] = stats_analysis.best_ks(KS_stats[cond][pin])
        
f = open('./Output/dualAdapt/KS_stats_tau_'+str(tau)+'.pckl', 'wb')
pickle.dump([KS_stats, wilcoxon_p], f)
f.close()

In [ ]:
tau = 0.1
pin = 0.9
cond = 'A'
wb_params = {}
c = np.zeros(K); scale = np.zeros(K)
for k in range(K):
    weights = A_matrices[cond][tau][pin][k][A_matrices[cond][tau][pin][k]>0]
    c[k], loc, scale[k] = stats.weibull_min.fit(weights, floc = 0)
wb_params['loc'] = np.mean(c)
wb_params['scale'] = np.std(c)

f = open('./Output/dualAdapt/weibull_fit_pin_'+str(pin)+'_tau_'+str(tau)+'_A.pckl', 'wb')
pickle.dump(wb_params, f)
f.close()

In [ ]:
# Compute the decay rate of the right tail of the weight distribution
res_PL = {}; PL_expon = {} 
for cond in cond_ls:
    res_PL[cond] = {}; PL_expon[cond] = {} 
    
    for tau in tau_ls:
        res_PL[cond][tau] = {}; PL_expon[cond][tau] = {} 
        for l, pin in enumerate(pin_ls):
            res_PL_temp = {}
            PL_expon_temp = []
            for k in range(K):
                # fit the right tail of the weight distribution with a linear function
                res_PL_temp[k] = stats_analysis.PL_MLE(A_matrices[cond][tau][pin][k], num_bins)

                # keep the decay rate if r2 of the linear fit is above the threshold
                if res_PL_temp[k][2] > r2_thresh:
                    PL_expon_temp.append(-res_PL_temp[k][1])
                    
            res_PL[cond][tau][pin] = pd.DataFrame.from_dict(res_PL_temp, orient='index', columns = ["intercept", "slope", "r2"])

            # keep the decay rates if more than half of the instances have r2 above the threshold
            if np.mean(res_PL[cond][tau][pin]['r2'] > r2_thresh) > 0.5:
                PL_expon[cond][tau][pin] = PL_expon_temp
                
f = open('./Output/dualAdapt/PL_fit.pckl', 'wb')
pickle.dump([res_PL, PL_expon], f)
f.close()

In [ ]:
# Import dictionaries of networks sampled at different steps in simulations for stationary analysis
tau = 0.1
pin_ls = [0.1, 0.3, 0.5, 0.7, 0.9]

A_dualAdapt = {}
for cond in cond_ls:
    A_dualAdapt[cond] = {}
    for pin in pin_ls:
        A_dualAdapt[cond][pin] = {}
        for k in range(K):       
            f = open('./Output_long/dualAdapt/'+cond+'_pin_'+str(pin)+'_eta_'+str(eta)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
            A_dualAdapt[cond][pin][k], _ = pickle.load(f)
            f.close()

In [ ]:
# Summary statistics
std = {}; skew = {}; kurt = {}
for cond in cond_ls:
    std[cond] = {}; skew[cond] = {}; kurt[cond] = {}
    for pin in pin_ls:
        std[cond][pin] = {}
        skew[cond][pin] = {}
        kurt[cond][pin] = {}
        for i, j in enumerate(A_dualAdapt[cond][0.1][0].keys()):
            std[cond][pin][j] = np.zeros(K)
            skew[cond][pin][j] = np.zeros(K)
            kurt[cond][pin][j] = np.zeros(K)
            
for cond in cond_ls:
    for pin in pin_ls:
        for k in range(K):
            for i, j in enumerate(A_dualAdapt[cond][0.1][0].keys()):
                std[cond][pin][j][k], skew[cond][pin][j][k], kurt[cond][pin][j][k] = stats_analysis.stats_summaries(A_dualAdapt[cond][pin][k][j])

f = open('./Output_long/dualAdapt/summary_stats_evol_tau_'+str(tau)+'.pckl', 'wb')
pickle.dump([std, skew, kurt], f)
f.close()

In [ ]:
# Connectivity metrics
cc = {}; ae = {}; mod = {}

for cond in cond_ls:
    cc[cond] = {}
    ae[cond] = {}
    mod[cond] = {} 
    for pin in pin_ls:
        cc[cond][pin] = {}
        ae[cond][pin] = {}
        mod[cond][pin] = {}
        for i, j in enumerate(A_dualAdapt[cond][0.1][0].keys()):
            cc[cond][pin][j] = np.zeros(K)
            ae[cond][pin][j] = np.zeros(K)
            mod[cond][pin][j] = np.zeros(K)

for cond in cond_ls:
    for pin in pin_ls:
        for k in range(K):
            for i, j in enumerate(A_dualAdapt[cond][0.1][0].keys()):
                A = A_dualAdapt[cond][pin][k][j].copy()
                cc[cond][pin][j][k] = indices.cluster_coefficient(A)
                ae[cond][pin][j][k] = indices.average_efficiency(A)
                mod[cond][pin][j][k], _ = directed_modularity.getModularityIndex(A)
                
f = open('./Output_long/dualAdapt/Topo_metrics_evol_tau_'+str(tau)+'.pckl', 'wb')
pickle.dump([cc, ae, mod], f)
f.close()  

# Convergent-divergent units

In [ ]:
# Import resulted networks at the end of simulations
tau_ls = [0.05, 0.1, 0.2, 0.5, 1.0]
A_matrices = {}
for cond in cond_ls:
    A_matrices[cond] = {}
    for tau in tau_ls:
        A_matrices[cond][tau] = {}
        for k in range(K):   
            f = open('./Output/cdUnits/'+cond+'_eta_'+str(eta)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
            A_matrices[cond][tau][k], _ = pickle.load(f)
            f.close()

In [ ]:
# Create null networks for comparison
# For each network generated by the model, the corresponding null network starts from the connectivity same to the initial random networks 
# but the weights are same to the final networks

f = open('./Output/inits.pckl', 'rb')
initial_mat = pickle.load(f)
f.close()

tau_rewire = 1.0 # tau_rewire same to the experiment
prand = 0.1 # proportion of random rewiring
final_step = 50 * 10000

null_inits = {}; null_A = {}
for cond in cond_ls:
    null_inits[cond] = {}
    null_A[cond] = {}
    for tau in tau_ls:
        null_inits[cond][tau] = {}
        null_A[cond][tau] = {}
        for k in range(K):
            print(cond + ', '+ str(tau) + ', ' +str(k))
            null_inits[cond][tau][k], null_A[cond][tau][k] = indices.rewire_only(A_matrices[cond][tau][k], initial_mat[k], 
            tau_rewire, tau, prand, final_step)            

f = open('./Output/cdUnits/inits_cd_null.pckl', 'wb')
pickle.dump(null_inits, f)
f.close()

f = open('./Output/cdUnits/cd_null.pckl', 'wb')
pickle.dump(null_A, f)
f.close()

In [ ]:
f = open('./Output/cdUnits/cd_null.pckl', 'rb')
null_A = pickle.load(f)
f.close()

In [ ]:
# Compute number of convergent and divergent hubs for simulated networks and null networks
binary_flag = True # define hubs by degree
thresh_ls = [5, 10, 15, 20, 25, 30] # threshold of hubs

con_num = {}; div_num = {}
con_ref = {}; div_ref = {}
for cond in cond_ls:
    con_num[cond] = {}; div_num[cond] = {}
    con_ref[cond] = {}; div_ref[cond] = {}
    for tau in tau_ls:
        con_num[cond][tau] = {}
        div_num[cond][tau] = {}
        con_ref[cond][tau] = {}
        div_ref[cond][tau] = {}
        for i, thresh in enumerate(thresh_ls):    
            con_num[cond][tau][thresh] = np.zeros(K)
            div_num[cond][tau][thresh] = np.zeros(K)
            con_ref[cond][tau][thresh] = np.zeros(K)
            div_ref[cond][tau][thresh] = np.zeros(K)            
            for k in range(K):
                con_num[cond][tau][thresh][k] = indices.hub_number(A_matrices[cond][tau][k], thresh, binary_flag, axisUsed=1)
                div_num[cond][tau][thresh][k] = indices.hub_number(A_matrices[cond][tau][k], thresh, binary_flag, axisUsed=0)

                con_ref[cond][tau][thresh][k] = indices.hub_number(null_A[cond][tau][k], thresh, binary_flag, axisUsed=1)
                div_ref[cond][tau][thresh][k] = indices.hub_number(null_A[cond][tau][k], thresh, binary_flag, axisUsed=0)

f = open('./Output/cdUnits/hubs.pckl', 'wb')
pickle.dump([con_num, con_ref, div_num, div_ref], f)
f.close()

In [ ]:
# Compute connectivity and obtain the cd units and intermediacte nodes for simulated networks and null networks
thresh = 15 # threshold of hubs
n = 100 # number of nodes
conn_mat = {}; conn_ref = {}
cd_pairs = {}; interms = {}
cd_pairs_ref = {}; interms_ref = {}
for cond in cond_ls:
    conn_mat[cond] = {}; cd_pairs[cond] = {}; interms[cond] = {}
    conn_ref[cond] = {}; cd_pairs_ref[cond] = {}; interms_ref[cond] = {}
    for tau in tau_ls:
        conn_mat[cond][tau] = np.zeros(K); conn_ref[cond][tau] = np.zeros(K)
        cd_pairs[cond][tau] = {}; cd_pairs_ref[cond][tau] = {}
        interms[cond][tau] = {}; interms_ref[cond][tau] = {}
        for k in range(K):
            conn_mat_temp = indices.connectedness(A_matrices[cond][tau][k])
            conn_mat[cond][tau][k] = np.sum(conn_mat_temp)/(n*(n-1))
            cd_pairs[cond][tau][k] = indices.cd_pairs(A_matrices[cond][tau][k], conn_mat_temp, thresh)
            interms[cond][tau][k] = indices.find_interm_subG(A_matrices[cond][tau][k], cd_pairs[cond][tau][k])

            conn_ref_temp = indices.connectedness(null_A[cond][tau][k])
            conn_ref[cond][tau][k] = np.sum(conn_ref_temp)/(n*(n-1))
            cd_pairs_ref[cond][tau][k] = indices.cd_pairs(null_A[cond][tau][k], conn_ref_temp, thresh)
            interms_ref[cond][tau][k] = indices.find_interm_subG(null_A[cond][tau][k], cd_pairs_ref[cond][tau][k])

f = open('./Output/cdUnits/connectedness.pckl', 'wb')
pickle.dump([conn_mat, conn_ref], f)
f.close()

f = open('./Output/cdUnits/cdPairs.pckl', 'wb')
pickle.dump([cd_pairs, cd_pairs_ref, interms, interms_ref], f)
f.close()

In [ ]:
# Compute number of cd units, average size, average density and average weight of intermediate subgraphs

cd_num = {}; subG_size = {}; subG_density = {}; subG_aveW = {}
cd_num_ref = {}; subG_size_ref = {}; subG_density_ref = {}; subG_aveW_ref = {}

for cond in cond_ls:
    cd_num[cond] = {}; subG_size[cond] = {}; subG_density[cond] = {}; subG_aveW[cond] = {}
    cd_num_ref[cond] = {}; subG_size_ref[cond] = {}; subG_density_ref[cond] = {}; subG_aveW_ref[cond] = {}
    for tau in tau_ls:
        cd_num[cond][tau] = np.zeros(K); subG_size[cond][tau] = np.zeros(K)
        subG_density[cond][tau] = np.zeros(K); subG_aveW[cond][tau] = np.zeros(K)
        cd_num_ref[cond][tau] = np.zeros(K); subG_size_ref[cond][tau] = np.zeros(K)
        subG_density_ref[cond][tau] = np.zeros(K); subG_aveW_ref[cond][tau] = np.zeros(K)
        for k in range(K):
            cd_num[cond][tau][k] = len(cd_pairs[cond][tau][k][0, ])
            cd_num_ref[cond][tau][k] = len(cd_pairs_ref[cond][tau][k][0, ])
         
            subG_size[cond][tau][k], subG_density[cond][tau][k], subG_aveW[cond][tau][k] = indices.interm_stats(A_matrices[cond][tau][k], interms[cond][tau][k])
            subG_size_ref[cond][tau][k], subG_density_ref[cond][tau][k], subG_aveW_ref[cond][tau][k] = indices.interm_stats(null_A[cond][tau][k], interms_ref[cond][tau][k])

f = open('./Output/cdUnits/interm_stats.pckl', 'wb')
pickle.dump([cd_num, cd_num_ref, subG_size, subG_size_ref, subG_density, subG_density_ref, subG_aveW, subG_aveW_ref], f)
f.close()

In [ ]:
# test if there is significant difference between the average density of intermediate subgraphs of simulated networks and that of null networks
p_density = {}
for s, cond in enumerate(cond_ls):
    p_density[cond] = {}
    for tau in tau_ls:
        res = stats.ttest_ind(indices.clean_group(subG_density[cond][tau]), indices.clean_group(subG_density_ref[cond][tau]))
        p_density[cond][tau] = res.pvalue

p_density

In [ ]:
# test if there is significant difference between the average weigt of intermediate subgraphs of simulated networks and that of null networks
p_aveW = {}
for s, cond in enumerate(cond_ls):
    p_aveW[cond] = {}

    for tau in tau_ls:
        res = stats.ttest_ind(indices.clean_group(subG_aveW[cond][tau]), indices.clean_group(subG_aveW_ref[cond][tau]))
        p_aveW[cond][tau] = res.pvalue

f = open('./Output/cdUnits/pvalues.pckl', 'wb')
pickle.dump([p_density, p_aveW], f)
f.close()

p_aveW

In [ ]:
# Import dictionaries of networks sampled at different steps in simulations for stationary analysis
tau_ls = [0.05, 0.1, 0.5]

A_matrices = {}
for cond in cond_ls:
    A_matrices[cond] = {}
    for tau in tau_ls:
        A_matrices[cond][tau] = {}
        for k in range(K):   
            f = open('./Output_long/cdUnits/'+cond+'_eta_'+str(eta)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
            A_matrices[cond][tau][k], _ = pickle.load(f)
            f.close()

In [ ]:
# Summary statistics
std = {}; skew = {}; kurt = {}
for cond in cond_ls:
    std[cond] = {}; skew[cond] = {}; kurt[cond] = {}
    for tau in tau_ls:
        std[cond][tau] = {}
        skew[cond][tau] = {}
        kurt[cond][tau] = {}
        for i, j in enumerate(A_matrices[cond][tau][0].keys()):
            std[cond][tau][j] = np.zeros(K)
            skew[cond][tau][j] = np.zeros(K)
            kurt[cond][tau][j] = np.zeros(K)
            
for cond in cond_ls:
    for tau in tau_ls:
        for k in range(K):
            for i, j in enumerate(A_matrices[cond][tau][k].keys()):
                std[cond][tau][j][k], skew[cond][tau][j][k], kurt[cond][tau][j][k] = stats_analysis.stats_summaries(A_matrices[cond][tau][k][j])

f = open('./Output_long/cdUnits/summary_stats_evol.pckl', 'wb')
pickle.dump([std, skew, kurt], f)
f.close()

In [ ]:
# Connectivity metrics
cc = {}; ae = {}; mod = {}

for cond in cond_ls:
    cc[cond] = {}
    ae[cond] = {}
    mod[cond] = {} 
    for tau in tau_ls:
        cc[cond][tau] = {}
        ae[cond][tau] = {}
        mod[cond][tau] = {}
        for i, j in enumerate(A_matrices[cond][tau][0].keys()):
            cc[cond][tau][j] = np.zeros(K)
            ae[cond][tau][j] = np.zeros(K)
            mod[cond][tau][j] = np.zeros(K)

for cond in cond_ls:
    for tau in tau_ls:
        for k in range(K):
            for i, j in enumerate(A_matrices[cond][tau][k].keys()):
                A = A_matrices[cond][tau][k][j].copy()
                cc[cond][tau][j][k] = indices.cluster_coefficient(A)
                ae[cond][tau][j][k] = indices.average_efficiency(A)
                mod[cond][tau][j][k], _ = directed_modularity.getModularityIndex(A)

f = open('./Output_long/cdUnits/Topo_metrics_evol.pckl', 'wb')
pickle.dump([cc, ae, mod], f)
f.close()  